# 🧪 9개 모델 비교 테스트

Baseline(kor_unsmile) + Fine-tuned 8개 모델을 게임 테스트 데이터로 비교합니다.

In [ ]:
import os, torch, pandas as pd, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import precision_recall_fscore_support, label_ranking_average_precision_score
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# 설정
MAX_LENGTH = 128
LABEL_NAMES = ["여성/가족", "남성", "성소수자", "인종/국적", "연령", "지역", "종교", "기타 혐오", "악플/욕설", "clean"]

# 비교할 모델 리스트
MODELS = {
    "kor_unsmile (Baseline)": "smilegate-ai/kor_unsmile",
    "LoRA v1 Game (KcELECTRA)": "./models/lora_game_kcelectra/merged_model",
    "LoRA v1 Tutorial (kcbert)": "./models/lora_tutorial_kcbert/merged_model",
    "LoRA v2 Game (KcELECTRA)": "./models/lora_game_kcelectra_v2/merged_model",
    "LoRA v2 Tutorial (kcbert)": "./models/lora_tutorial_kcbert_v2/merged_model",
    "Full FT v1 Game (KcELECTRA)": "./models/full_game_kcelectra/best_model",
    "Full FT v1 Tutorial (kcbert)": "./models/full_tutorial_kcbert/best_model",
    "Full FT v2 Game (KcELECTRA)": "./models/full_game_kcelectra_v2/best_model",
    "Full FT v2 Tutorial (kcbert)": "./models/full_tutorial_kcbert_v2/best_model",
}

In [ ]:
# 테스트 데이터 로드
test_df = pd.read_csv("./data/game_test.tsv", sep='\t')
print(f"✅ 테스트 데이터: {len(test_df)}건")
test_df.head()

In [ ]:
def evaluate_model(model_name, model_path, test_df):
    """모델 평가 함수"""
    print(f"\n🔄 평가 중: {model_name}")
    
    try:
        # 모델 로드
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSequenceClassification.from_pretrained(model_path).to(DEVICE)
        model.eval()
        
        # 예측
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for idx, row in test_df.iterrows():
                inputs = tokenizer(
                    row['문장'], 
                    padding='max_length', 
                    truncation=True, 
                    max_length=MAX_LENGTH,
                    return_tensors='pt'
                ).to(DEVICE)
                
                outputs = model(**inputs)
                probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]
                all_preds.append(probs)
                
                labels = [float(row[col]) for col in LABEL_NAMES]
                all_labels.append(labels)
        
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        
        # 메트릭 계산
        preds_binary = (all_preds > 0.5).astype(int)
        labels_int = all_labels.astype(int)
        
        lrap = label_ranking_average_precision_score(all_labels, all_preds)
        _, abuse_r, abuse_f1, _ = precision_recall_fscore_support(
            labels_int[:,8], preds_binary[:,8], average='binary', zero_division=0
        )
        _, clean_r, clean_f1, _ = precision_recall_fscore_support(
            labels_int[:,9], preds_binary[:,9], average='binary', zero_division=0
        )
        
        print(f"  LRAP: {lrap:.4f}, Abuse Recall: {abuse_r:.4f}, Abuse F1: {abuse_f1:.4f}")
        
        return {
            'model': model_name,
            'lrap': lrap,
            'abuse_recall': abuse_r,
            'abuse_f1': abuse_f1,
            'clean_recall': clean_r,
            'clean_f1': clean_f1
        }
    
    except Exception as e:
        print(f"  ❌ 오류: {e}")
        return {
            'model': model_name,
            'lrap': None,
            'abuse_recall': None,
            'abuse_f1': None,
            'clean_recall': None,
            'clean_f1': None
        }

In [ ]:
# 모든 모델 평가
results = []

for model_name, model_path in MODELS.items():
    result = evaluate_model(model_name, model_path, test_df)
    results.append(result)
    
    # GPU 메모리 정리
    torch.cuda.empty_cache()

print("\n✅ 평가 완료!")

In [ ]:
# 결과 정리
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('abuse_recall', ascending=False)
results_df

In [ ]:
# 결과 저장
os.makedirs('./results', exist_ok=True)
results_df.to_csv('./results/comparison_results.csv', index=False)
print("✅ 결과 저장: ./results/comparison_results.csv")

In [ ]:
# 시각화
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Abuse Recall 비교
valid_results = results_df.dropna(subset=['abuse_recall'])
colors = ['red' if 'Baseline' in m else 'steelblue' for m in valid_results['model']]

axes[0].barh(valid_results['model'], valid_results['abuse_recall'], color=colors)
axes[0].set_xlabel('Abuse Recall')
axes[0].set_title('Abuse Recall 비교 (높을수록 좋음)')
axes[0].axvline(x=valid_results[valid_results['model'].str.contains('Baseline')]['abuse_recall'].values[0], 
                color='red', linestyle='--', label='Baseline')

# LRAP 비교
axes[1].barh(valid_results['model'], valid_results['lrap'], color=colors)
axes[1].set_xlabel('LRAP')
axes[1].set_title('LRAP 비교 (높을수록 좋음)')

plt.tight_layout()
plt.savefig('./results/comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ 차트 저장: ./results/comparison_chart.png")

In [ ]:
# 개선율 계산
baseline_abuse = results_df[results_df['model'].str.contains('Baseline')]['abuse_recall'].values[0]

print("\n📊 Baseline 대비 개선율")
print("="*50)

for _, row in results_df.iterrows():
    if row['abuse_recall'] is not None and 'Baseline' not in row['model']:
        improvement = ((row['abuse_recall'] - baseline_abuse) / baseline_abuse) * 100
        sign = '+' if improvement > 0 else ''
        print(f"{row['model']}: {sign}{improvement:.1f}%")